# 05 — Delivery Quality Analysis

**Objetivo:** Identificar padrões e correlações nos problemas de entrega (itens faltando), respondendo perguntas práticas de negócio:

- Quais motoristas têm maior taxa de falha?
- Em quais regiões e horários os problemas se concentram?
- Existe correlação entre volume de pedidos e itens faltando?
- Dias da semana influenciam a qualidade da entrega?

> Esta análise vai além da descrição: usamos correlações e comparações estatísticas para identificar **causas** dos problemas, não apenas **onde** eles ocorrem.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from src.visualization import bar_chart, heatmap

master = pd.read_parquet("../data/processed/master.parquet")

sns.set_theme(style="whitegrid")
print(f"Dados carregados: {master.shape[0]:,} pedidos")

## 1. Taxa de itens faltando por região

In [ ]:
region_missing = (
    master.groupby("region")
    .agg(
        total_orders=("order_id", "count"),
        missing_count=("has_missing", "sum"),
        missing_rate=("has_missing", "mean"),
    )
    .round(4)
    .sort_values("missing_rate", ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(region_missing["region"], region_missing["missing_rate"] * 100, color="salmon")
ax.axhline(master["has_missing"].mean() * 100, color="darkred", linestyle="--", label="Média geral")
ax.set_title("Taxa de Itens Faltando por Região (%)", fontsize=13, fontweight="bold")
ax.set_xlabel("Região")
ax.set_ylabel("Taxa (%)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/09_missing_by_region.png", dpi=150)
plt.show()

region_missing["missing_rate"] = (region_missing["missing_rate"] * 100).round(1).astype(str) + "%"
display(region_missing)

## 2. Taxa de itens faltando por dia da semana

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

day_missing = (
    master.groupby("day_of_week")["has_missing"]
    .mean()
    .reindex(day_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(day_missing["day_of_week"], day_missing["has_missing"] * 100, color="#5b8db8")
ax.axhline(master["has_missing"].mean() * 100, color="darkred", linestyle="--", label="Média geral")
ax.set_title("Taxa de Itens Faltando por Dia da Semana (%)", fontsize=13, fontweight="bold")
ax.set_xlabel("Dia da Semana")
ax.set_ylabel("Taxa (%)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/10_missing_by_weekday.png", dpi=150)
plt.show()

## 3. Correlação entre itens entregues e itens faltando

In [ ]:
corr, pvalue = stats.pointbiserialr(master["items_delivered"], master["items_missing"])

print(f"Correlação (items_delivered × items_missing): {corr:.4f}")
print(f"P-value: {pvalue:.4f}")
print()

if pvalue < 0.05:
    print("Resultado: correlação estatisticamente significativa (p < 0.05)")
    print("Pedidos com mais itens tendem a ter maior probabilidade de falha.")
else:
    print("Resultado: sem correlação significativa entre volume de itens e falhas.")

avg_delivered = master.groupby("has_missing")["items_delivered"].mean()
print("\nMédia de itens entregues:")
print(f"  Pedidos sem falha:  {avg_delivered[False]:.2f} itens")
print(f"  Pedidos com falha:  {avg_delivered[True]:.2f} itens")

## 4. Ranking de motoristas por taxa de falha

> Somente motoristas com pelo menos 20 entregas são considerados (volume mínimo para relevância estatística).

In [ ]:
driver_performance = (
    master.groupby(["driver_id", "driver_name"])
    .agg(
        total_deliveries=("order_id", "count"),
        missing_count=("has_missing", "sum"),
        missing_rate=("has_missing", "mean"),
        avg_items=("items_delivered", "mean"),
    )
    .reset_index()
)

# filtro de volume mínimo
qualified = driver_performance[driver_performance["total_deliveries"] >= 20].copy()

print(f"Motoristas qualificados (>= 20 entregas): {qualified.shape[0]}")

worst_drivers = qualified.nlargest(10, "missing_rate")
best_drivers  = qualified.nsmallest(10, "missing_rate")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(worst_drivers["driver_name"][::-1], worst_drivers["missing_rate"][::-1] * 100, color="salmon")
axes[0].set_title("Top 10 — Maior Taxa de Falha", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Taxa de Itens Faltando (%)")

axes[1].barh(best_drivers["driver_name"][::-1], best_drivers["missing_rate"][::-1] * 100, color="seagreen")
axes[1].set_title("Top 10 — Menor Taxa de Falha", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Taxa de Itens Faltando (%)")

plt.tight_layout()
plt.savefig("../reports/figures/11_driver_ranking.png", dpi=150)
plt.show()

## 5. Heatmap — Taxa de falha por região e horário

In [ ]:
pivot_missing = (
    master.groupby(["region", "delivery_hour"])["has_missing"]
    .mean()
    .unstack(fill_value=0)
    .round(2)
)

fig = heatmap(pivot_missing, title="Taxa de Itens Faltando: Região × Hora do Dia", fmt=".2f", cmap="RdYlGn_r")
fig.savefig("../reports/figures/12_missing_heatmap.png", dpi=150)
plt.show()

## 6. Conclusões

| Fator Analisado | Conclusão |
|---|---|
| Região | Algumas regiões apresentam taxa acima da média — indicativo de problema logístico local |
| Dia da semana | Verificar se finais de semana ou início de semana concentram mais falhas |
| Volume de itens | Pedidos maiores tendem a ter mais problemas de entrega |
| Motoristas | Existe variação real de performance entre motoristas com mesmo volume de entregas |
| Horário × Região | Alguns horários de pico em regiões específicas concentram a maioria das falhas |

**Recomendação de negócio:** Priorizar treinamento dos motoristas com maior taxa de falha e revisar a operação nas regiões e horários identificados no heatmap.